# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The rule, in plain words: "A page is worth reviewing first if it's still getting found in search, but it's trending down and hasn't been touched in a while." That's the stale-visible archetype from ML-02/03, turned into a rule a human can check by eye.

Reason codes (every scored page gets exactly one, so the queue is auditable):

* `stale_declining_visible` — old update, trending down, meaningful impressions → top priority
* `declining_visible_fresh` — trending down and visible, but recently updated (something else is wrong, not staleness)
* `stale_visible_stable` — old update, decent visibility, not yet declining (early warning, not urgent)
* `low_priority` — doesn't meet the visibility bar, nothing to act on yet

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# --- flyrank-data gotchas applied ---
# avg_position == 0 means "no data", not rank zero — don't let it silently pass a "good position" filter
df["has_position_data"] = df["avg_position"] > 0

# rate columns are already ×100 percentages (ctr, engagement_rate, etc.) — no rescale needed here,
# we aren't using them in the rule itself, just noting it so nobody double-scales later

# --- conditions, in plain words first ---
visibility_bar = df["impressions_90d"].quantile(0.5)   # "meaningful" = above the median page
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= visibility_bar).astype(int)
declining = (df["trend_direction"] == "down").astype(int)

# --- transparent score: readable on purpose, no fitted weights ---
df["score"] = declining * visible * (1 + stale) * df["impressions_90d"]
# +1 to stale (not multiply-by-zero) so declining+visible pages still rank even if fresh,
# but stale ones get boosted above them — keeps the rule additive-flavored, not all-or-nothing

# --- reason codes ---
def reason_code(row):
    if row["trend_direction"] == "down" and row["impressions_90d"] >= visibility_bar:
        return "stale_declining_visible" if row["days_since_last_update"] >= 180 else "declining_visible_fresh"
    if row["impressions_90d"] >= visibility_bar and row["days_since_last_update"] >= 180:
        return "stale_visible_stable"
    return "low_priority"

df["reason_code"] = df.apply(reason_code, axis=1)

# --- rank ---
ranked = df.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
ranked[[
    "content_id", "client_id", "score", "reason_code",
    "impressions_90d", "sessions_90d", "trend_direction", "trend_pct",
    "days_since_last_update", "content_type", "avg_position", "has_position_data"
]].to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Reason code counts:")
print(df["reason_code"].value_counts())
print("\nTop 5 rows:")
print(ranked[["content_id", "score", "reason_code", "impressions_90d", "trend_direction", "days_since_last_update"]].head())

Reason code counts:
reason_code
low_priority               21085
declining_visible_fresh     8900
stale_declining_visible       14
stale_visible_stable           1
Name: count, dtype: int64

Top 5 rows:
             content_id   score              reason_code  impressions_90d  \
0  content_5fe46e04994d  517715  declining_visible_fresh           517715   
1  content_8c19996aa890  509252  declining_visible_fresh           509252   
2  content_4c36c775b818  463103  declining_visible_fresh           463103   
3  content_1a9e894be2e2  416180  declining_visible_fresh           416180   
4  content_2c2606c5d176  347399  declining_visible_fresh           347399   

  trend_direction  days_since_last_update  
0            down                     104  
1            down                      20  
2            down                      20  
3            down                      22  
4            down                     104  


In [2]:
# Does update recency correlate with traffic size?
print(df.groupby(pd.qcut(df["impressions_90d"], 4, duplicates="drop"))["days_since_last_update"].median())

impressions_90d
(0.999, 81.0]          20.0
(81.0, 731.0]          22.0
(731.0, 3615.25]       22.0
(3615.25, 517715.0]    25.0
Name: days_since_last_update, dtype: float64


/tmp/ipykernel_1949/18514847.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby(pd.qcut(df["impressions_90d"], 4, duplicates="drop"))["days_since_last_update"].median())


In [4]:
priority_order = ["stale_declining_visible", "declining_visible_fresh", "stale_visible_stable", "low_priority"]
df["reason_code"] = pd.Categorical(df["reason_code"], categories=priority_order, ordered=True)
ranked = df.sort_values(["reason_code", "impressions_90d"], ascending=[True, False]).reset_index(drop=True)

ranked[[
    "content_id", "client_id", "reason_code",
    "impressions_90d", "sessions_90d", "trend_direction", "trend_pct",
    "days_since_last_update", "content_type", "avg_position"
]].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(ranked[["content_id", "reason_code", "impressions_90d", "days_since_last_update"]].head(20))

              content_id              reason_code  impressions_90d  \
0   content_cf56e2e2e282  stale_declining_visible            61678   
1   content_7368877ea310  stale_declining_visible            59472   
2   content_1bfaa38ff26c  stale_declining_visible            25715   
3   content_0a91db491d14  stale_declining_visible            13299   
4   content_5feee3994adb  stale_declining_visible             7812   
5   content_c2d929d83eaa  stale_declining_visible             7558   
6   content_b16bd7307b39  stale_declining_visible             4590   
7   content_fe16a55cd13d  stale_declining_visible             4556   
8   content_ecb6215e79fd  stale_declining_visible             4429   
9   content_928af3e22c80  stale_declining_visible             1697   
10  content_e3ff1b093148  stale_declining_visible             1408   
11  content_7f116ae1f6f5  stale_declining_visible              954   
12  content_77d4d5930e5e  stale_declining_visible              828   
13  content_72496874

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

This is the part building-baselines is most insistent on — hand-review, not just trust the score. Run this to pull the fields you need per row, then fill the note/confidence columns yourself (this is genuinely a by-hand step, not one to fully automate):

In [5]:
top20 = ranked.head(20)[[
    "content_id", "client_id", "reason_code", "score",
    "impressions_90d", "sessions_90d", "trend_pct",
    "days_since_last_update", "content_type", "word_count", "avg_position"
]]
top20.to_csv("work/outputs/top20_for_review.csv", index=False)
print(top20)

              content_id          client_id              reason_code   score  \
0   content_cf56e2e2e282  client_7f2253d7e2  stale_declining_visible  123356   
1   content_7368877ea310  client_7f2253d7e2  stale_declining_visible  118944   
2   content_1bfaa38ff26c  client_7f2253d7e2  stale_declining_visible   51430   
3   content_0a91db491d14  client_7f2253d7e2  stale_declining_visible   26598   
4   content_5feee3994adb  client_7f2253d7e2  stale_declining_visible   15624   
5   content_c2d929d83eaa  client_7f2253d7e2  stale_declining_visible   15116   
6   content_b16bd7307b39  client_7f2253d7e2  stale_declining_visible    9180   
7   content_fe16a55cd13d  client_7f2253d7e2  stale_declining_visible    9112   
8   content_ecb6215e79fd  client_7f2253d7e2  stale_declining_visible    8858   
9   content_928af3e22c80  client_7f2253d7e2  stale_declining_visible    3394   
10  content_e3ff1b093148  client_d029fa3a95  stale_declining_visible    2816   
11  content_7f116ae1f6f5  client_9400f1b

For each of the 20, write (as a markdown table or a column you add to this dataframe):

* **Action** — protect / improve / rewrite / merge / prune / monitor, picked from the archetype-to-action mapping in your Lane 3 brief
* **Reason code** — already attached, just carry it over
* **Confidence note** — one honest line, e.g. "high confidence — big impression base, clear decline" vs "low confidence — impressions just above median, could be noise" vs "flag — `has_position_data` is False, position claims unreliable here"
* **What would make it wrong** — the specific thing that would flip your action, e.g. "wrong if the decline is seasonal, not structural — check same page last year" or "wrong if this page was intentionally deprecated (check for a redirect already in place)"

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

1. **Client concentration in the top bucket**. Rows 0–9 and 12 (11 of the top 14) all belong to `client_7f2253d7e2`. That's not necessarily wrong — this client's inventory may genuinely be more stale than others — but check before trusting it:

In [6]:
print(df[df["reason_code"] == "stale_declining_visible"]["client_id"].value_counts())

client_id
client_7f2253d7e2    11
client_4ec9599fc2     1
client_d029fa3a95     1
client_9400f1b21c     1
Name: count, dtype: int64


If this baseline is meant to serve a portfolio-wide priority queue, one client dominating the top raises a real question: is this reflecting genuine urgency, or just that one client has systematically older content across the board (which a per-client normalized rule would need to account for, e.g. rank within-client percentile rather than raw values)? Name this as a limitation either way.

2. **Missing** `word_count` **on your highest-impression fresh-declining pages**. Rows 14, 17, 18 (517k, 416k, 347k impressions — your biggest pages) all have `word_count = NaN`. Per the flyrank-data skill, missingness follows `content_type`, not randomness — but these are all `content_type == "keyword article"`, same as the rows with word_count. So this isn't the known category-level gap; it's worth a direct check:

In [7]:
print(df[df["content_id"].isin(["content_5fe46e04994d","content_1a9e894be2e2","content_2c2606c5d176"])][["content_type","word_count","char_count"]])

          content_type  word_count  char_count
6653   keyword article         NaN         NaN
13537  keyword article         NaN         NaN
29879  keyword article         NaN         NaN


Don't let a blind fillna hide this — if `char_count` is also NaN, that's a real gap in the record for your biggest pages, worth flagging rather than silently treating as "short content."

3. **A genuine rule-design finding, not a bug: rows 15–17 are your most urgent pages, but rank below the stale bucket**. They sit at position 2.3–2.5 (page-1, top-3) and are still losing 33–45% of traffic — that's a much bigger absolute cost than a page-2 stale article, because they started from peak visibility. Your reason-code-first ranking (Option B) correctly captured "stale AND declining" as the named rule, but it means a top-3-ranked page actively bleeding traffic ranks below a stale page-2 article. That's worth stating honestly in your write-up as a real tradeoff of the rule as designed — not something to silently patch, since patching it would be moving the goalposts after seeing the output, which the baseline skill explicitly warns against.

**Leakage Check:**

In [8]:
used_in_score = ["days_since_last_update", "impressions_90d", "trend_direction"]
print("Fields used in scoring:", used_in_score)
print("Fields used only for review (not scoring):",
      ["trend_pct", "avg_position", "word_count", "content_type", "sessions_90d"])
print("provider_used / model_used touched:", False)

Fields used in scoring: ['days_since_last_update', 'impressions_90d', 'trend_direction']
Fields used only for review (not scoring): ['trend_pct', 'avg_position', 'word_count', 'content_type', 'sessions_90d']
provider_used / model_used touched: False


Everything used is backward-looking (`*_90d`, `*_last_30d`/`*_prev_30d`-derived `trend_direction`) — no future window, no product-decision flags. Clean.

**Weak picks:** (1) 11 of 14 top-priority pages belong to a single client — the rule measures absolute staleness, not staleness relative to a client's own baseline, so it isn't yet safe for a cross-client comparison without normalization. (2) The three highest-impression pages in the fresh-declining bucket (517k, 416k, 347k impressions) have both `word_count` and `char_count` missing — a genuine content-metadata gap, not a category pattern, meaning we can't verify content length before recommending action on our most urgent pages. (3) The rule can't distinguish "urgent decline from a strong position" (rows 15–17: page-1 rank, still losing 33–45% traffic) from "steady decline from a weak position" — both currently rank by the same reason-code tier, though the former likely costs more in absolute terms.

**Leakage check**: confirmed clean. Scoring used only `days_since_last_update`, `impressions_90d`, and `trend_direction` — all backward-looking, trailing-window fields. No product/tooling flags (`provider_used`, `model_used`) and no future-window data were used anywhere in the rule.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.